In [ ]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm


In [ ]:
# Update this path to your dataset location
DATASET_PATH = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/raw"


# Check how many folders exist
folders = [
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
]

print(f"✅ Found {len(folders)} call folders")

In [ ]:
# Test with first folder
sample_folder = os.path.join(DATASET_PATH, folders[0])
print(folders[0])

# Load each file
with open(f"{sample_folder}/meeting-info.json") as f:
    meeting = json.load(f)

with open(f"{sample_folder}/summary.json") as f:
    summary = json.load(f)

with open(f"{sample_folder}/transcript.json") as f:
    transcript = json.load(f)

# Preview
print("📋 MEETING INFO:")
print(f"   Title    : {meeting['title']}")
print(f"   Duration : {meeting['duration']} mins")
print(f"   Start    : {meeting['startTime']}")
print(f"   Emails   : {meeting['allEmails']}")
print("\n📊 SUMMARY:")
print(f"   Sentiment : {summary['overallSentiment']}")
print(f"   Score     : {summary['sentimentScore']}")
print(f"   Topics    : {summary['topics']}")
print(f"   Actions   : {len(summary['actionItems'])} items")

print("\n💬 TRANSCRIPT TYPE:", type(transcript))
print("   Preview   :", str(transcript)[:200])

In [ ]:
# Look at first 3 sentences
sentences = transcript['data']

print(f"📝 Total sentences: {len(sentences)}")
print(f"\n🗣️ First 3 sentences:")
for s in sentences[:3]:
    print(f"   [{s['speaker_name']}] ({s['sentimentType']}) → {s['sentence']}")
    
print(f"\n🎭 Unique sentiment types in this call:")
sentiments = set(s['sentimentType'] for s in sentences)
print(f"   {sentiments}")

print(f"\n👥 Unique speakers:")
speakers = set(s['speaker_name'] for s in sentences)
print(f"   {speakers}")

In [ ]:
def load_all_calls(dataset_path):
    folders = [
        f for f in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, f))
    ]

    data = []
    errors = []

    for folder in tqdm(folders, desc="Loading calls"):
        folder_path = os.path.join(dataset_path, folder)

        try:
            # Load files
            with open(f"{folder_path}/meeting-info.json") as f:
                meeting = json.load(f)
            with open(f"{folder_path}/summary.json") as f:
                summary = json.load(f)
            with open(f"{folder_path}/transcript.json") as f:
                transcript = json.load(f)

            # Extract sentences
            sentences = transcript.get("data", [])
            full_text = " ".join([s["sentence"] for s in sentences])

            # Sentence-level sentiment counts
            sent_counts = {"positive": 0, "negative": 0, "neutral": 0}
            for s in sentences:
                t = s.get("sentimentType", "neutral")
                if t in sent_counts:
                    sent_counts[t] += 1

            record = {
                # Meeting Info
                "meeting_id"        : meeting.get("meetingId"),
                "title"             : meeting.get("title"),
                "duration_mins"     : meeting.get("duration"),
                "start_time"        : meeting.get("startTime"),
                "participants"      : meeting.get("allEmails", []),
                "num_participants"  : len(meeting.get("allEmails", [])),

                # Summary
                "summary"           : summary.get("summary"),
                "topics"            : summary.get("topics", []),
                "sentiment"         : summary.get("overallSentiment"),
                "sentiment_score"   : summary.get("sentimentScore"),
                "action_items"      : summary.get("actionItems", []),
                "num_action_items"  : len(summary.get("actionItems", [])),
                "key_moments"       : summary.get("keyMoments", []),

                # Transcript
                "full_text"         : full_text,
                "num_sentences"     : len(sentences),
                "positive_sents"    : sent_counts["positive"],
                "negative_sents"    : sent_counts["negative"],
                "neutral_sents"     : sent_counts["neutral"],
                "speakers"          : list(set(s["speaker_name"] for s in sentences)),
                "num_speakers"      : len(set(s["speaker_name"] for s in sentences)),
            }

            data.append(record)

        except Exception as e:
            errors.append({"folder": folder, "error": str(e)})

    print(f"\n✅ Loaded    : {len(data)} calls")
    print(f"❌ Errors    : {len(errors)} calls")
    if errors:
        for e in errors:
            print(f"   {e['folder']} → {e['error']}")

    return pd.DataFrame(data)


# Run it
df = load_all_calls(DATASET_PATH)
print(f"\n📊 DataFrame shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

In [ ]:
##  Detect the call type 

def classify_call_type(title):
    title_lower = title.lower()

    if any(word in title_lower for word in [
        "support", "case #", "issue", "ticket",
        "billing", "incident", "help"
    ]):
        return "Customer Support"

    elif any(word in title_lower for word in [
        "sync", "standup", "stand-up", "internal",
        "planning", "escalation", "retrospective",
        "review", "team", "engineering", "debrief"
    ]):
        return "Internal"

    else:
        return "External"


df["call_type"] = df["title"].apply(classify_call_type)

print("📞 Call Type Distribution:")
print(df["call_type"].value_counts())
print()
print("📊 Sentiment Distribution:")
print(df["sentiment"].value_counts())
print()
print("⏱️ Avg Duration by Call Type:")
print(df.groupby("call_type")["duration_mins"].mean().round(1))

In [ ]:
# Save to CSV for use in other notebooks
df.to_csv("/Users/ashishjain/Documents/assignment/transcript-intelligence/data/processed/all_calls.csv", index=False)
print("✅ Saved → data/processed/all_calls.csv")

# Quick preview
df[["title", "call_type", "sentiment",
    "sentiment_score", "duration_mins",
    "num_sentences", "num_action_items"]].head(10)

In [ ]:
# Check actual column names
print(df.columns.tolist())
print(f"\nShape: {df.shape}")